In [ ]:
import pandas as pd
import requests
from pathlib import Path

# --- Configuration ---
TEAM_WATCHLIST_PATH = Path('../Data/processed/regional_watchlist_teams.csv')

def test_aes_endpoint():
    # 1. Load a target from your Watchlist
    if not TEAM_WATCHLIST_PATH.exists():
        print("Team watchlist not found. Please pin a team first.")
        return
        
    df_teams = pd.read_csv(TEAM_WATCHLIST_PATH)
    if df_teams.empty:
        print("Watchlist is empty.")
        return
        
    # Grab the first team in your list for the trial
    target_team = df_teams.iloc[0]
    team_name = target_team['Name']
    
    # Ensure TeamId is an integer (removing any decimals if they sneaked in)
    team_id = int(target_team['TeamId']) 
    
    print(f"--- MISSION START: API RECONNAISSANCE ---")
    print(f"Target: {team_name}")
    print(f"Team ID: {team_id}")
    
    # 2. Construct the URL
    url = f"https://advancedeventsystems.com/rankings/{team_id}"
    print(f"Endpoint: {url}")
    
    # Note: We use a "User-Agent" header to mimic a standard web browser. 
    # Many sports APIs block automated Python scripts if this is missing.
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    try:
        # 3. Execute the Request
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code == 200:
            print("\n[SUCCESS] Connection established.")
            
            # 4. Analyze the Payload
            content_type = response.headers.get('Content-Type', '')
            
            if 'application/json' in content_type:
                print("Payload Type: JSON")
                data = response.json()
                print("\n--- JSON Data Snippet ---")
                # Print the highest level keys to understand the structure
                print(f"Keys: {list(data.keys())}")
                # Print a small snippet of the actual data
                print(str(data)[:500] + "...") 
            else:
                print("Payload Type: HTML / Text")
                print("\n--- HTML Code Snippet ---")
                # Print the first 500 characters to see the tags
                print(response.text[:500])
        else:
            print(f"\n[ERROR] Server rejected request. Status Code: {response.status_code}")
            
    except Exception as e:
        print(f"\n[CRITICAL ERROR] Connection failed: {e}")

# Execute the trial
test_aes_endpoint()


In [ ]:
import pandas as pd
import requests
from IPython.display import display

# Use the specific team ID you found
TEAM_ID = 3472
url = f"https://advancedeventsystems.com/rankings/{TEAM_ID}"

print(f"--- MISSION START: HTML EXTRACTION ---")
print(f"Target URL: {url}")

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

try:
    response = requests.get(url, headers=headers, timeout=10)
    
    if response.status_code == 200:
        # Pandas read_html automatically parses all <table> tags in an HTML string
        tables = pd.read_html(response.text)
        
        print(f"\n[SUCCESS] Found {len(tables)} data tables on the page.")
        
        # Display the first few rows of each table found
        for i, table_df in enumerate(tables):
            print(f"\n--- Table {i} Preview ---")
            display(table_df.head())
            
except ValueError as ve:
    # A ValueError from read_html usually means no <table> tags were found
    print("\n[INFO] No static HTML tables were found on the page.")
    print("This likely means AES loads the ranking data dynamically using JavaScript.")
    print(f"Error Details: {ve}")
    
except Exception as e:
    print(f"\n[CRITICAL ERROR] Connection or parsing failed: {e}")


In [ ]:
import pandas as pd
import requests
from pathlib import Path
from IPython.display import display

TEAM_WATCHLIST_PATH = Path('../Data/processed/regional_watchlist_teams.csv')
BASE_API = "https://advancedeventsystems.com/api/ranking"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept': 'application/json'
}

print("--- MISSION START: WATCHLIST API INTEGRATION ---")

if not TEAM_WATCHLIST_PATH.exists():
    print("[!] Team watchlist not found. Please pin a team first.")
else:
    df_teams = pd.read_csv(TEAM_WATCHLIST_PATH)
    
    if df_teams.empty:
        print("[!] Watchlist is empty.")
    else:
        # 1. Grab the first team from your watchlist
        target_team = df_teams.iloc[0]
        team_name = target_team['Name']
        team_id = int(target_team['TeamId']) 
        
        print(f"\nTarget: {team_name}")
        print(f"Local CSV Team ID: {team_id}")
        
        # 2. Ping the Base API Endpoint
        print(f"\nPinging AES API: {BASE_API}/{team_id}")
        response = requests.get(f"{BASE_API}/{team_id}", headers=headers)
        
        if response.status_code == 200:
            print("\n[SUCCESS] 1:1 Mapping Confirmed! The CSV TeamId works in the API.")
            data = response.json()
            
            # Print a few key stats
            api_name = data.get('TeamName', data.get('Name', 'Unknown'))
            rank = data.get('Rank', 'N/A')
            points = data.get('TotalPoints', data.get('Points', 'N/A'))
            
            print("-" * 30)
            print("AES LIVE DATA MATCH:")
            print(f"Name in AES: {api_name}")
            print(f"Current Rank: {rank}")
            print(f"Total Points: {points}")
            print("-" * 30)
            
            # 3. Pull the Finishes Table
            print("\nPulling Tournament Finishes...")
            finish_resp = requests.get(f"{BASE_API}/{team_id}/finishes", headers=headers)
            if finish_resp.status_code == 200:
                df_finishes = pd.json_normalize(finish_resp.json())
                display(df_finishes.head(3))
                
        elif response.status_code == 404:
            print("\n[FAILED] Error 404: The API did not recognize this ID.")
            print("This means the TeamId in the CSV is different from the AES Ranking ID.")
        else:
            print(f"\n[ERROR] Server returned Status: {response.status_code}")


In [ ]:
import pandas as pd
import requests
from pathlib import Path
from IPython.display import display

TEAM_WATCHLIST_PATH = Path('../Data/processed/regional_watchlist_teams.csv')
BASE_API = "https://advancedeventsystems.com/api/ranking"
MAIN_PAGE = "https://advancedeventsystems.com/rankings"

# --- BROSWER EMULATION HEADERS ---
# These headers trick the firewall into thinking we are a real Chrome user
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Connection': 'keep-alive',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-origin',
}

print("--- MISSION START: STEALTH API INTEGRATION ---")

if not TEAM_WATCHLIST_PATH.exists():
    print("[!] Team watchlist not found.")
else:
    df_teams = pd.read_csv(TEAM_WATCHLIST_PATH)
    
    if df_teams.empty:
        print("[!] Watchlist is empty.")
    else:
        target_team = df_teams.iloc[0]
        team_name = target_team['Name']
        team_id = int(target_team['TeamId']) 
        
        print(f"\nTarget: {team_name} | ID: {team_id}")
        
        # 1. Establish a Session
        # This allows us to hold onto security cookies between requests
        session = requests.Session()
        
        try:
            # 2. Recon: Visit the main page first to get cookies
            print(f"Bypassing Firewall: Acquiring session cookies from {MAIN_PAGE}/{team_id}...")
            session.get(f"{MAIN_PAGE}/{team_id}", headers={'User-Agent': headers['User-Agent']}, timeout=10)
            
            # 3. Add the Referer to our headers (Critical for bypassing 403s)
            headers['Referer'] = f"{MAIN_PAGE}/{team_id}"
            
            # 4. Strike: Hit the JSON API endpoint
            api_url = f"{BASE_API}/{team_id}"
            print(f"Pinging Data Payload: {api_url}")
            response = session.get(api_url, headers=headers, timeout=10)
            
            if response.status_code == 200:
                print("\n[SUCCESS] Firewall bypassed. Payload secured.")
                data = response.json()
                
                # Extract and display stats
                api_name = data.get('TeamName', data.get('Name', 'Unknown'))
                rank = data.get('Rank', 'N/A')
                points = data.get('TotalPoints', data.get('Points', 'N/A'))
                
                print("-" * 30)
                print("AES LIVE DATA MATCH:")
                print(f"Name in AES: {api_name}")
                print(f"Current Rank: {rank}")
                print(f"Total Points: {points}")
                print("-" * 30)
                
            elif response.status_code == 403:
                print("\n[FAILED] Error 403: Firewall is still blocking the request.")
                print("AES might be using advanced bot-protection (like Cloudflare Turnstile).")
            elif response.status_code == 404:
                print("\n[FAILED] Error 404: The ID does not exist in the ranking database.")
            else:
                print(f"\n[ERROR] Server returned Status: {response.status_code}")
                
        except Exception as e:
            print(f"Connection Error: {e}")


In [ ]:
import pandas as pd
import requests
import json
from pathlib import Path

TEAM_WATCHLIST_PATH = Path('../Data/processed/regional_watchlist_teams.csv')
BASE_API = "https://advancedeventsystems.com/api/ranking"
MAIN_PAGE = "https://advancedeventsystems.com/rankings"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Connection': 'keep-alive',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-origin',
}

df_teams = pd.read_csv(TEAM_WATCHLIST_PATH)
team_id = int(df_teams.iloc[0]['TeamId']) 

session = requests.Session()
session.get(f"{MAIN_PAGE}/{team_id}", headers={'User-Agent': headers['User-Agent']})
headers['Referer'] = f"{MAIN_PAGE}/{team_id}"

response = session.get(f"{BASE_API}/{team_id}", headers=headers)

if response.status_code == 200:
    data = response.json()
    print("--- RAW JSON KEYS ---")
    print(list(data.keys()))
    
    print("\n--- RAW JSON PREVIEW ---")
    # Print the first 500 characters of the formatted JSON to see the structure
    print(json.dumps(data, indent=2)[:800])
else:
    print(f"Error: {response.status_code}")

    

In [ ]:
import pandas as pd
import requests
import json
import time
from pathlib import Path
from IPython.display import display

# =============================================================================
# CONFIGURATION
# =============================================================================
TEAM_ID = 185651
BASE_API = "https://advancedeventsystems.com/api/ranking"
MAIN_PAGE = "https://advancedeventsystems.com/rankings"

# Create a dedicated folder for this team's raw data
OUTPUT_DIR = Path(f'../Data/processed/AES_{TEAM_ID}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Stealth Headers
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json',
    'Referer': f"{MAIN_PAGE}/{TEAM_ID}"
}

# The four endpoints we want to explore
endpoints = {
    "Base_Info": "",
    "Events_Matches": "/events",
    "Tournament_Finishes": "/finishes",
    "Team_Members": "/members"
}

print(f"--- MISSION START: FULL DATA EXTRACTION FOR ID {TEAM_ID} ---")
print(f"Output Directory: {OUTPUT_DIR}\n")

session = requests.Session()

# 1. Establish Session Cookies (Firewall Bypass)
print("Establishing secure session...")
session.get(f"{MAIN_PAGE}/{TEAM_ID}", headers={'User-Agent': headers['User-Agent']})

# 2. Iterate through all endpoints
for name, suffix in endpoints.items():
    target_url = f"{BASE_API}/{TEAM_ID}{suffix}"
    print(f"\n[+] Extracting: {name.replace('_', ' ')}")
    print(f"    URL: {target_url}")
    
    try:
        response = session.get(target_url, headers=headers, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            
            # Save the raw JSON to a file for your manual inspection
            file_path = OUTPUT_DIR / f"{name.lower()}.json"
            with open(file_path, 'w') as f:
                json.dump(data, f, indent=4)
            
            print(f"    [SUCCESS] Saved to {file_path.name}")
            
            # --- DISPLAY DATA PREVIEWS ---
            if isinstance(data, list):
                # If the payload is a list (like events, finishes, members), show as a table
                df = pd.json_normalize(data)
                print(f"    Rows Extracted: {len(df)}")
                print(f"    Columns Available: {list(df.columns)}")
                display(df.head(3))  # Show first 3 rows
            elif isinstance(data, dict):
                # If it's a single dictionary (like Base Info), just show the keys
                print(f"    Keys Available: {list(data.keys())}")
            
        elif response.status_code == 403:
            print("    [FAILED] 403 Forbidden - Firewall blocked the request.")
        elif response.status_code == 404:
            print("    [FAILED] 404 Not Found - Endpoint does not exist for this team.")
        else:
            print(f"    [FAILED] Status Code: {response.status_code}")
            
    except Exception as e:
        print(f"    [ERROR] Connection failed: {e}")
        
    # Throttle to be polite to their servers
    time.sleep(1)

print("\n--- EXTRACTION COMPLETE ---")
print("You can now open the JSON files in your editor to see every single piece of data they hold.")


In [ ]:
import pandas as pd
import json
from pathlib import Path
from IPython.display import display

# =============================================================================
# CONFIGURATION
# =============================================================================
TEAM_ID = 185651
EVENTS_FILE = Path(f'../Data/processed/AES_{TEAM_ID}/events_matches.json')

print(f"--- MISSION START: EVENTS DATA INSPECTION ({TEAM_ID}) ---")

if not EVENTS_FILE.exists():
    print(f"[!] Could not find {EVENTS_FILE}. Please run the extraction script first.")
else:
    # 1. Load the raw JSON data
    with open(EVENTS_FILE, 'r') as f:
        events_data = json.load(f)
        
    print(f"[SUCCESS] Loaded {len(events_data)} match records.")
    
    # 2. Normalize the JSON into a DataFrame
    # json_normalize flattens nested dictionaries (e.g., opponent.name -> opponent_name)
    df_events = pd.json_normalize(events_data)
    
    # Print all available columns so we know exactly what AES tracks
    print("\n--- ALL AVAILABLE COLUMNS ---")
    for col in df_events.columns:
        print(f"- {col}")
        
    # 3. Create a "Clean" View
    # Note: Because AES data changes slightly by region/event, these column names 
    # might need adjusting based on the printout above. I am guessing the common ones.
    
    print("\n--- ATTEMPTING TO BUILD A CLEAN MATCH HISTORY ---")
    
    try:
        # We look for the most likely column names for Opponent and Result
        opponent_col = [c for c in df_events.columns if 'opponent' in c.lower() and 'name' in c.lower()]
        result_col = [c for c in df_events.columns if 'result' in c.lower() or 'won' in c.lower()]
        date_col = [c for c in df_events.columns if 'date' in c.lower() or 'time' in c.lower()]
        
        # Grab the first match we found for each to build a clean table
        cols_to_show = []
        if date_col: cols_to_show.append(date_col[0])
        if opponent_col: cols_to_show.append(opponent_col[0])
        if result_col: cols_to_show.append(result_col[0])
        
        if cols_to_show:
            clean_view = df_events[cols_to_show].copy()
            display(clean_view.head(10))
        else:
            print("Could not automatically guess the column names for Opponent and Result.")
            display(df_events.head(3))
            
    except Exception as e:
        print(f"Could not build clean view: {e}")
        display(df_events.head(3))

        

In [ ]:
import pandas as pd
import json
from pathlib import Path
from IPython.display import display

TEAM_ID = 185651
EVENTS_FILE = Path(f'../Data/processed/AES_{TEAM_ID}/events_matches.json')

print(f"--- MISSION START: TOURNAMENT SCHEDULE ({TEAM_ID}) ---")

with open(EVENTS_FILE, 'r') as f:
    events_data = json.load(f)
    
df_events = pd.json_normalize(events_data)

# Extract the most relevant columns for a human-readable schedule
schedule_cols = [
    'eventId', 
    'name', 
    'startDate', 
    'endDate', 
    'address.city', 
    'address.state.abbreviation'
]

# Check which of these columns actually exist in the dataframe
available_cols = [col for col in schedule_cols if col in df_events.columns]

df_schedule = df_events[available_cols].copy()

# Clean up the column names for the display
df_schedule = df_schedule.rename(columns={
    'eventId': 'Event_ID',
    'name': 'Tournament_Name',
    'startDate': 'Start_Date',
    'endDate': 'End_Date',
    'address.city': 'City',
    'address.state.abbreviation': 'State'
})

# Format the dates to remove the 'T00:00:00' timestamp
if 'Start_Date' in df_schedule.columns:
    df_schedule['Start_Date'] = pd.to_datetime(df_schedule['Start_Date']).dt.strftime('%Y-%m-%d')
if 'End_Date' in df_schedule.columns:
    df_schedule['End_Date'] = pd.to_datetime(df_schedule['End_Date']).dt.strftime('%Y-%m-%d')

# Sort chronologically
if 'Start_Date' in df_schedule.columns:
    df_schedule = df_schedule.sort_values(by='Start_Date')

display(df_schedule)


In [ ]:
import pandas as pd
import json
from pathlib import Path
from IPython.display import display

# =============================================================================
# CONFIGURATION
# =============================================================================
TEAM_ID = 185651
FINISHES_FILE = Path(f'../Data/processed/AES_{TEAM_ID}/tournament_finishes.json')

print(f"--- MISSION START: TOURNAMENT FINISHES INSPECTION ({TEAM_ID}) ---")

if not FINISHES_FILE.exists():
    print(f"[!] Could not find {FINISHES_FILE}. Please run the extraction script first.")
else:
    # Load and normalize the JSON
    with open(FINISHES_FILE, 'r') as f:
        finishes_data = json.load(f)
        
    df_finishes = pd.json_normalize(finishes_data)
    
    print(f"[SUCCESS] Loaded {len(df_finishes)} finish records.")
    
    print("\n--- ALL AVAILABLE COLUMNS ---")
    for col in df_finishes.columns:
        print(f"- {col}")
        
    print("\n--- RAW DATA PREVIEW (First 5 rows) ---")
    display(df_finishes.head())


In [ ]:
import pandas as pd
import json
from pathlib import Path
from IPython.display import display

TEAM_ID = 185651
FINISHES_FILE = Path(f'../Data/processed/AES_{TEAM_ID}/tournament_finishes.json')

with open(FINISHES_FILE, 'r') as f:
    finishes_data = json.load(f)
    
df_finishes = pd.json_normalize(finishes_data)

# 1. Clean up column names
clean_finishes = df_finishes.rename(columns={
    'event.name': 'Tournament',
    'event.startDate': 'Date',
    'eventDivision': 'Division',
    'finishRank': 'Rank',
    'divisionSize': 'Field_Size'
}).copy()

# 2. Format the Date
clean_finishes['Date'] = pd.to_datetime(clean_finishes['Date']).dt.strftime('%Y-%m-%d')

# 3. Calculate Performance Metrics
# Create a readable "X out of Y" string
clean_finishes['Placement'] = clean_finishes['Rank'].astype(str) + " of " + clean_finishes['Field_Size'].astype(str)

# Calculate what Top % of the field they finished in
clean_finishes['Top_Percent'] = (clean_finishes['Rank'] / clean_finishes['Field_Size'] * 100).round(1)
clean_finishes['Top_Percent'] = "Top " + clean_finishes['Top_Percent'].astype(str) + "%"

# 4. Final Display Formatting
final_cols = ['Date', 'Tournament', 'Division', 'Placement', 'Top_Percent']
df_display = clean_finishes[final_cols].sort_values(by='Date')

print(f"--- TOURNAMENT PERFORMANCE DASHBOARD ({TEAM_ID}) ---")
display(df_display)


In [ ]:
import pandas as pd
import json
from pathlib import Path
from IPython.display import display

# =============================================================================
# CONFIGURATION
# =============================================================================
TEAM_ID = 185651
MEMBERS_FILE = Path(f'../Data/processed/AES_{TEAM_ID}/team_members.json')

print(f"--- MISSION START: ROSTER INSPECTION ({TEAM_ID}) ---")

if not MEMBERS_FILE.exists():
    print(f"[!] Could not find {MEMBERS_FILE}. Ensure the extraction script ran successfully.")
else:
    # 1. Load the raw JSON data
    with open(MEMBERS_FILE, 'r') as f:
        members_data = json.load(f)
        
    # 2. Normalize into a DataFrame
    df_members = pd.json_normalize(members_data)
    
    if df_members.empty:
        print("[INFO] The roster file is empty. The club has not made this roster public on AES.")
    else:
        print(f"[SUCCESS] Loaded {len(df_members)} roster records (Players/Staff).")
        
        print("\n--- ALL AVAILABLE COLUMNS ---")
        for col in df_members.columns:
            print(f"- {col}")
            
        print("\n--- RAW ROSTER PREVIEW (First 5 rows) ---")
        display(df_members.head())


In [ ]:
import pandas as pd
import json
from pathlib import Path
from IPython.display import display

# =============================================================================
# CONFIGURATION
# =============================================================================
TEAM_ID = 185651
MEMBERS_FILE = Path(f'../Data/processed/AES_{TEAM_ID}/team_members.json')

with open(MEMBERS_FILE, 'r') as f:
    members_data = json.load(f)
    
df_members = pd.json_normalize(members_data)

# 1. Select and Rename the important columns
df_roster = df_members[[
    'memberFirstName', 
    'memberLastName', 
    'jerseyNumber', 
    'position', 
    'gradYear', 
    'teamUserType.isPlayer',
    'teamUserType.displayName'
]].copy()

df_roster = df_roster.rename(columns={
    'memberFirstName': 'First_Name',
    'memberLastName': 'Last_Name',
    'jerseyNumber': 'Jersey',
    'position': 'Position',
    'gradYear': 'Class',
    'teamUserType.isPlayer': 'Is_Player',
    'teamUserType.displayName': 'Role'
})

# 2. Clean up the Numbers (Remove the .0 decimals)
# Fill missing values with -1 temporarily, convert to integer, then replace -1 with blank
df_roster['Jersey'] = df_roster['Jersey'].fillna(-1).astype(int).astype(str).replace('-1', '')
df_roster['Class'] = df_roster['Class'].fillna(-1).astype(int).astype(str).replace('-1', '')

# Clean up empty position strings
df_roster['Position'] = df_roster['Position'].fillna('')

# 3. Split into Players and Staff
df_players = df_roster[df_roster['Is_Player'] == True].copy()
df_staff = df_roster[df_roster['Is_Player'] == False].copy()

# Sort Players by Jersey Number (requires converting back to numeric for proper sorting)
df_players['Jersey_Sort'] = pd.to_numeric(df_players['Jersey'], errors='coerce').fillna(999)
df_players = df_players.sort_values(by='Jersey_Sort').drop(columns=['Jersey_Sort', 'Is_Player', 'Role'])

# Sort Staff by Role
df_staff = df_staff[['First_Name', 'Last_Name', 'Role']].sort_values(by='Role')

print(f"--- TEAM ROSTER: ID {TEAM_ID} ---")
print("\n[ COACHING STAFF ]")
display(df_staff)

print("\n[ ACTIVE PLAYERS ]")
display(df_players)


In [ ]:
import pandas as pd
import requests
import json
from IPython.display import display

# =============================================================================
# CONFIGURATION
# =============================================================================
TEAM_ID = 185651
EVENT_ID = 40917  # 7th Annual AC Invitational

MATCHES_API_URL = f"https://advancedeventsystems.com/api/ranking/{TEAM_ID}/events/{EVENT_ID}/matches"
MAIN_PAGE = "https://advancedeventsystems.com/rankings"

# Stealth Headers for Firewall Bypass
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept': 'application/json',
    'Referer': f"{MAIN_PAGE}/{TEAM_ID}"
}

print(f"--- MISSION START: LIVE MATCH EXTRACTION ---")
print(f"Targeting Event {EVENT_ID} for Team {TEAM_ID}...")

session = requests.Session()

# 1. Establish Session Context
session.get(f"{MAIN_PAGE}/{TEAM_ID}", headers={'User-Agent': headers['User-Agent']})

try:
    # 2. Execute Data Extraction
    response = session.get(MATCHES_API_URL, headers=headers, timeout=10)
    
    if response.status_code == 200:
        matches_data = response.json()
        print(f"\n[SUCCESS] Extracted {len(matches_data)} live match records.")
        
        # 3. Flatten the JSON
        df_live_matches = pd.json_normalize(matches_data)
        
        print("\n--- ALL AVAILABLE COLUMNS ---")
        # Print columns alphabetically to make it easier to scan for "code"
        for col in sorted(df_live_matches.columns):
            print(f"- {col}")
            
        print("\n--- RAW MATCH DATA PREVIEW ---")
        display(df_live_matches.head(3))
        
    elif response.status_code == 403:
        print("\n[FAILED] 403 Forbidden. Firewall blocked the request.")
    else:
        print(f"\n[FAILED] Server returned Status: {response.status_code}")

except Exception as e:
    print(f"\n[ERROR] Connection failed: {e}")


In [ ]:
import pandas as pd
import requests
from IPython.display import display

TEAM_ID = 185651
EVENT_ID = 40917  # 7th Annual AC Invitational

MATCHES_API_URL = f"https://advancedeventsystems.com/api/ranking/{TEAM_ID}/events/{EVENT_ID}/matches"
MAIN_PAGE = "https://advancedeventsystems.com/rankings"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept': 'application/json',
    'Referer': f"{MAIN_PAGE}/{TEAM_ID}"
}

print(f"--- MISSION START: UNWRAPPING MATCH DATA ---")

session = requests.Session()
session.get(f"{MAIN_PAGE}/{TEAM_ID}", headers={'User-Agent': headers['User-Agent']})

try:
    response = session.get(MATCHES_API_URL, headers=headers, timeout=10)
    
    if response.status_code == 200:
        raw_data = response.json()
        
        # --- THE FIX: UNWRAP THE PAYLOAD ---
        # If AES wrapped the list in a dictionary under the key 'value'
        if isinstance(raw_data, dict) and 'value' in raw_data:
            matches_data = raw_data['value']
        # Sometimes they wrap it in 'data'
        elif isinstance(raw_data, dict) and 'data' in raw_data:
            matches_data = raw_data['data']
        else:
            matches_data = raw_data
            
        print(f"\n[SUCCESS] Unwrapped {len(matches_data)} live match records.")
        
        # Flatten the unwrapped list
        df_live_matches = pd.json_normalize(matches_data)
        
        print("\n--- ALL AVAILABLE COLUMNS ---")
        for col in sorted(df_live_matches.columns):
            print(f"- {col}")
            
        print("\n--- RAW MATCH DATA PREVIEW ---")
        display(df_live_matches.head())
        
except Exception as e:
    print(f"\n[ERROR] Connection failed: {e}")


In [ ]:
import pandas as pd
import requests
from IPython.display import display
import ast

TEAM_ID = 185651
EVENT_ID = 40917  

MATCHES_API_URL = f"https://advancedeventsystems.com/api/ranking/{TEAM_ID}/events/{EVENT_ID}/matches"
MAIN_PAGE = "https://advancedeventsystems.com/rankings"

headers = {
    'User-Agent': 'Mozilla/5.0',
    'Accept': 'application/json',
    'Referer': f"{MAIN_PAGE}/{TEAM_ID}"
}

print(f"--- MISSION START: PARSING SET SCORES ---")
session = requests.Session()
session.get(f"{MAIN_PAGE}/{TEAM_ID}", headers={'User-Agent': headers['User-Agent']})

response = session.get(MATCHES_API_URL, headers=headers, timeout=10)
    
if response.status_code == 200:
    raw_data = response.json()
    matches_data = raw_data.get('value', raw_data.get('data', raw_data))
    
    df_live = pd.json_normalize(matches_data)
    
    # Helper function to extract sets from the JSON list
    def extract_sets(scores_list):
        # Handle cases where the list is empty or invalid
        if not isinstance(scores_list, list) or len(scores_list) == 0:
            return pd.Series([None, None, None, None, None, None])
            
        s1_t, s1_o, s2_t, s2_o, s3_t, s3_o = [None] * 6
        
        # Set 1
        if len(scores_list) > 0:
            s1_t = scores_list[0].get('teamScore')
            s1_o = scores_list[0].get('opponentScore')
        # Set 2
        if len(scores_list) > 1:
            s2_t = scores_list[1].get('teamScore')
            s2_o = scores_list[1].get('opponentScore')
        # Set 3 (If it went to a deciding set)
        if len(scores_list) > 2:
            s3_t = scores_list[2].get('teamScore')
            s3_o = scores_list[2].get('opponentScore')
            
        return pd.Series([s1_t, s1_o, s2_t, s2_o, s3_t, s3_o])

    # Apply the function to create new columns
    df_live[['Set1_Team', 'Set1_Opp', 'Set2_Team', 'Set2_Opp', 'Set3_Team', 'Set3_Opp']] = df_live['scores'].apply(extract_sets)
    
    # Create the clean display
    clean_display = df_live[[
        'opponentName', 
        'opponentCode', 
        'matchOutcomeType.displayName', 
        'Set1_Team', 'Set1_Opp', 
        'Set2_Team', 'Set2_Opp', 
        'Set3_Team', 'Set3_Opp'
    ]].copy()
    
    clean_display.rename(columns={'matchOutcomeType.displayName': 'Result'}, inplace=True)
    
    print("\n--- FINAL CLEAN MATCH HISTORY ---")
    display(clean_display)


In [ ]:
import sys
from pathlib import Path

# Add the src folder to the Python path so Jupyter can find your new module
sys.path.append(str(Path('../src').resolve()))

# Import the class you just built
from api_client import AESClient

# Initialize the client
client = AESClient()

# Execute a massive raw data pull with ONE line of code
client.get_all_team_data(123957)